In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np
from collections import deque
from stable_baselines3.common.callbacks import CallbackList
from sb3_contrib import MaskablePPO
from game.plot_game import render_text
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
import os
from datetime import datetime
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state, encode_state_cnn_modern
from sb3_utils.callbacks import AveragedMetricsCallback, SaveEveryNTimestepsCallback, UnfreezeCallback
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from game.plot_game import render_text_with_blocks

In [3]:
class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space
        
        dummy_obs, _ = self.raw_env.reset()
        grid, blocks = encode_state_cnn_modern(dummy_obs)


        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.observation_space = spaces.Dict({
            'grid': spaces.Box(low=0.0, high=1.0, shape=(grid.shape), dtype=np.float32),
            'blocks': spaces.Box(low=0.0, high=1.0, shape=(blocks.shape), dtype=np.float32),
        })

        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        grid, blocks = encode_state_cnn_modern(obs_dict)
        return {'grid': grid.astype(np.float32), 'blocks': blocks.astype(np.float32)}, {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        grid, blocks = encode_state_cnn_modern(obs_dict)
        return {'grid': grid.astype(np.float32), 'blocks': blocks.astype(np.float32)}, reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [4]:
raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3)
wrapped_env = DiscreteActionWrapper(raw_env)
def mask_fn(env):
    return env.raw_env.game.compute_action_mask()
masked_env = ActionMasker(wrapped_env, mask_fn)
check_env(masked_env, warn=True) 

/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation blocks has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation grid has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [5]:
class BlockPuzzleFeaturesExtended(BaseFeaturesExtractor):
    """
    A custom feature extractor that:
      - Takes obs = Dict({
            'grid':   Tensor of shape [batch, H, W],
            'blocks': Tensor of shape [batch, N, P] (e.g., N=3)
        })
      - Applies a small CNN to 'grid'
      - Applies a shared MLP to each of the N one-hot rows in 'blocks'
      - Concatenates CNN features + concatenated_block_embeddings → output vector
    """
    def __init__(self, observation_space: gym.spaces.Dict,  cnn_channels=[16,32,32,64,64],  block_embedding_dim=64):
        super().__init__(observation_space, features_dim=1)  # we’ll override features_dim below

        # 1) Extract shapes from the space:
        grid_shape   = observation_space.spaces['grid'].shape   # (H, W)
        blocks_shape = observation_space.spaces['blocks'].shape # (N, P) -> N is num_blocks
        H, W         = grid_shape
        self.num_block_slots, P = blocks_shape # N (e.g., 3)

        # 2) Build the CNN for 'grid' → produce a flat vector
        self.cnn = nn.Sequential(
            nn.Conv2d(1,   cnn_channels[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(cnn_channels[0], cnn_channels[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(cnn_channels[1], cnn_channels[2], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(cnn_channels[2], cnn_channels[3], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(cnn_channels[3], cnn_channels[4], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        # 3) Build the shared block‐embedding MLP (P → block_embedding_dim)
        self.block_mlp = nn.Sequential(
            nn.Linear(P,      128),
            nn.ReLU(),
            nn.Linear(128,    block_embedding_dim),
            nn.ReLU(),
        )

        # Figure out the CNN’s output size:
        with torch.no_grad():
            # H, W already extracted
            dummy_grid_obs = torch.zeros(1, 1, H, W) # Add channel dimension for CNN
            cnn_out_features = self.cnn(dummy_grid_obs)
            cnn_out_size = cnn_out_features.shape[1]

        # Calculate the size of the concatenated block embeddings
        # Each of the self.num_block_slots blocks will be embedded to block_embedding_dim.
        # Then, these N embeddings are concatenated.
        concatenated_block_features_size = self.num_block_slots * block_embedding_dim # MODIFIED

        # 4) Final joint MLP to combine [ CNN_out || concatenated_block_embeddings ]
        joint_input_size = cnn_out_size + concatenated_block_features_size # MODIFIED
        self.joint_net = nn.Sequential(
            nn.Linear(joint_input_size, 256), # Input size updated
            nn.ReLU(),
            nn.Linear(256,               128),
            nn.ReLU(),
        )

        # 5) Tell SB3 how large the final feature vector is:
        self._features_dim = 128 # This is the output of self.joint_net

    def forward(self, observations: dict) -> torch.Tensor:
        """
        observations is a dict with:
          - 'grid':   torch.FloatTensor of shape (batch, H, W)
          - 'blocks': torch.FloatTensor of shape (batch, N, P) (e.g., N=3)
        We need to:
          1) Unsqueeze 'grid' → (batch, 1, H, W) → run through CNN → (batch, cnn_out_size)
          2) Flatten 'blocks' as (batch*N, P), run through block_mlp → (batch*N, block_embedding_dim),
             then reshape → (batch, N, block_embedding_dim)
          3) Reshape/Flatten the N block embeddings to concatenate them → (batch, N * block_embedding_dim)
          4) Concatenate the grid features and concatenated block features
             → (batch, cnn_out_size + N * block_embedding_dim)
          5) Run through joint_net → (batch, self._features_dim)
        """
        # 1) CNN on 'grid'
        grid = observations['grid']            # shape = (batch, H, W), float32
        x    = grid.unsqueeze(1)               # → (batch, 1, H, W)
        x    = self.cnn(x)                     # → (batch, cnn_out_size)

        # 2) Embed the N blocks
        blocks = observations['blocks']        # → (batch, N, P)
        B, N_slots, P_dim = blocks.shape       # B=batch, N_slots (e.g.,3), P_dim=one-hot size
        
        # Ensure N_slots matches self.num_block_slots used during initialization
        assert N_slots == self.num_block_slots, \
            f"Mismatch in number of blocks: expected {self.num_block_slots}, got {N_slots}"

        blocks_flat = blocks.view(B * N_slots, P_dim) # → (batch*N, P)
        e = self.block_mlp(blocks_flat)               # → (batch*N, block_embedding_dim)
        e = e.view(B, N_slots, -1)                    # → (batch, N, block_embedding_dim)

        # 3) Concatenate block embeddings (instead of summing)
        #    Reshape from (batch, N, block_embedding_dim) to (batch, N * block_embedding_dim)
        e_concat = e.reshape(B, -1)                   # MODIFIED: → (batch, N * block_embedding_dim)

        # 4) Concatenate grid features and concatenated block features
        joint = torch.cat([x, e_concat], dim=1)       # MODIFIED: use e_concat
                                                      # → (batch, cnn_out_size + N * block_embedding_dim)
        # 5) Pass through joint_net
        return self.joint_net(joint)                  # → (batch, self._features_dim)

In [6]:
def lr_warmup_schedule(progress_remaining) -> float:
    """
    SB3 passes in `progress_remaining` which goes from 1.0 → 0.0 over the entire learn() call.
    We want:
      - first 10% (progress 1.0 → 0.9): lr goes 1e-5 → 1e-3
      - remaining 90% (progress 0.9 → 0.0): lr goes 1e-3 → 1e-4
    """
    max_lr = 1e-3
    init_lr = 1e-5
    final_lr = 1e-4
    warmup_percent = 0.10  # 5% of the training time is warm-up

    # Calculate how far we are into training [0.0 .. 1.0]
    t = 1.0 - progress_remaining

    if t < warmup_percent:
        # warm-up phase: 0 → 0.05
        return init_lr + (max_lr - init_lr) * (t / warmup_percent)
    else:
        # decay phase: 0.05 → 1.0
        decay_t = (t - warmup_percent) / (1.0 - warmup_percent)  # remap [0.05..1.0] → [0..1]
        return max_lr + (final_lr - max_lr) * decay_t

In [7]:
def make_env():
    def _init():
        raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3)
        wrapped_env = DiscreteActionWrapper(raw_env)

        def mask_fn(env):
            return env.raw_env.game.compute_action_mask()

        masked_env = ActionMasker(wrapped_env, mask_fn)
        monitored_env = Monitor(masked_env)
        return monitored_env

    return _init

n_envs = 48
vec_env = SubprocVecEnv([make_env() for _ in range(n_envs)])

policy_kwargs = dict(
    features_extractor_class  = BlockPuzzleFeaturesExtended,
    features_extractor_kwargs = dict(
        cnn_channels        = [16, 32, 32, 64, 64],   # or whatever you prefer
        block_embedding_dim = 64,
    ),
)

model = MaskablePPO(
    policy            = "MultiInputPolicy",
    env               = vec_env,
    learning_rate     = lr_warmup_schedule,
    n_steps           = 256,
    batch_size        = 4096,
    gamma             = 0.90,
    device            = "cuda",
    verbose           = 1,
    tensorboard_log   = "./sb3_logs/",
    policy_kwargs     = policy_kwargs,
)

callback_list = CallbackList([
    AveragedMetricsCallback(), 
    SaveEveryNTimestepsCallback(save_freq=250_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
])

load_weights = False
weight_path = "sb3_block_ppo_cnn_modern_masked_50M"
if load_weights:
    model.set_parameters(weight_path, exact_match=False)


model.learn(total_timesteps=100_000_000, callback=callback_list, tb_log_name="PPO_CNN_MODERN_MASKED_DEEPER_FIXED")
model.save("sb3_block_ppo_cnn_modern_masked_deeper_fixed")

Using cuda device
Logging to ./sb3_logs/PPO_CNN_MODERN_MASKED_DEEPER_FIXED_2
-----------------------------------
| custom/              |          |
|    cleared_lines_avg | 1.18     |
|    invalid_moves_avg | 0        |
|    move_count_avg    | 17.8     |
| rollout/             |          |
|    ep_len_mean       | 17.8     |
|    ep_rew_mean       | -8.08    |
| time/                |          |
|    fps               | 3611     |
|    iterations        | 1        |
|    time_elapsed      | 3        |
|    total_timesteps   | 12288    |
-----------------------------------
-------------------------------------------
| custom/                 |               |
|    cleared_lines_avg    | 1.38          |
|    invalid_moves_avg    | 0             |
|    move_count_avg       | 17.5          |
| rollout/                |               |
|    ep_len_mean          | 17.5          |
|    ep_rew_mean          | -7.83         |
| time/                   |               |
|    fps               

KeyboardInterrupt: 

In [8]:
model.save("sb3_block_ppo_cnn_modern_masked_deeper_concat")

In [ ]:
# callback_list = CallbackList([
#     AveragedMetricsCallback(), 
#     SaveEveryNTimestepsCallback(save_freq=1_000_000, save_path="./sb3_model_saves/", name="sb3_block_ppo_mlp_masked_rewards2"),
#     SaveEveryNTimestepsCallback(save_freq=100_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
# ])

# for i in range(0, 50):
#     if i != 0:
#         backup_path = "crash_backup_save/backup_save__"
#         model.set_parameters(backup_path, exact_match=True)

#     try:
#         model.learn(total_timesteps=30_000_001, callback=callback_list, tb_log_name="PPO_MLP_MASKED")
#     except Exception as e:
#         pass

In [9]:

single_env = DiscreteActionWrapper(BlockPuzzleEnv(width=8, height=10, num_blocks=3))
obs, _ = single_env.reset()
done = False
total_reward = 0
step = 0

while not done:
    # Get valid action mask
    mask = single_env.raw_env.game.compute_action_mask()
        
    # Use model.predict with action_masks for MaskablePPO
    action, _ = model.predict(obs, action_masks=mask, deterministic=True)
    
    obs, reward, terminated, truncated, info = single_env.step(action)
    total_reward += reward
    done = terminated or truncated
    step += 1
    
    game_grid = single_env.raw_env.game.grid.get_game_grid()
    game_blocks = [block.grid() for block in single_env.raw_env.game.block_queue]

    render_text_with_blocks(game_grid, game_blocks)
    print()


print(f"Evaluation finished in {step} steps, total reward: {total_reward}")


□ □ □ □ □ □ □ □ | ■ □ □ □  ■ ■ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  ■ ■ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ |
■ □ □ □ □ □ □ □ |
■ ■ □ □ □ □ □ □ |
□ ■ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |

□ □ □ □ □ □ □ □ | ■ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ |
■ □ □ □ □ □ ■ ■ |
■ ■ □ □ □ □ ■ ■ |
□ ■ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |

□ □ □ □ □ □ □ □ | □ ■ □ □  ■ ■ ■ □  ■ ■ ■ ■
□ □ □ □ □ □ □ □ | ■ ■ □ □  □ ■ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | ■ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ |
■ □ □ □ □ ■ ■ ■ |
■ ■ □ □ □ □ ■ ■ |
□ ■ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |

□ □ □ □ □ □ □ □ | □ ■ □ □  ■ ■ ■ □  □ □ □ □
□ □ □ □ □ □ □ □ | ■ ■ □ □  □ ■ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | ■ □ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □

In [ ]:
game_grid